In [1]:
!apt-get update
!apt install chromium-chromedriver
!pip install selenium

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt' is not recognized as an internal or external command,
operable program or batch file.


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\60107\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install selenium webdriver-manager pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\60107\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from webdriver_manager.microsoft import EdgeChromiumDriverManager

def scrape_with_selenium(url, output_csv='facebook_comments2.csv'):
    print("Preparing Edge Browser Settings...")
    
    edge_options = EdgeOptions()
    # KEEP THIS COMMENTED OUT so you can see the browser
    # edge_options.add_argument("--headless=new") 
    edge_options.add_argument("--no-sandbox")
    edge_options.add_argument("--disable-dev-shm-usage")
    edge_options.add_argument("--disable-gpu")
    edge_options.add_argument("--disable-blink-features=AutomationControlled") 
    
    # --- THE MAGIC BULLET: PERSISTENT PROFILE ---
    # This creates a folder on your C: drive to remember you are logged in
    edge_options.add_argument(r"user-data-dir=C:\SeleniumEdgeProfile") 
    # --------------------------------------------
    
    edge_options.add_experimental_option('excludeSwitches', ['enable-logging'])

    try:
        print("Launching Edge Browser...")
        service = EdgeService(EdgeChromiumDriverManager().install())
        driver = webdriver.Edge(service=service, options=edge_options)
        
        # Go directly to the target post
        print(f"Navigating to {url}...")
        driver.get(url)
        
        # --- HUMAN INTERVENTION PAUSE ---
        print("\n" + "="*50)
        print("ACTION REQUIRED:")
        print("1. Look at the Edge browser.")
        print("2. If Facebook asks you to log in, take your time and do it.")
        print("3. Ensure the comments are fully visible on the screen.")
        print("="*50 + "\n")
        
        # The script will freeze right here forever until you hit Enter
        input("Press ENTER in this text box right here when you are ready to scrape...")
        
        print("Extracting elements...")
        comment_elements = driver.find_elements(By.CSS_SELECTOR, "div[dir='auto']")
        
        scraped_data = []
        for idx, elem in enumerate(comment_elements):
            text = elem.text.strip()
            if text and len(text) > 5: 
                scraped_data.append({
                    'id': idx + 1,
                    'message': text
                })
                
        print(f"Extracted {len(scraped_data)} potential comment text blocks.")
        
        if scraped_data:
            df = pd.DataFrame(scraped_data)
            df.to_csv(output_csv, index=False, encoding='utf-8')
            print(f"\n[SUCCESS] Saved data to {output_csv}")
            display(df.head())
        else:
            print("\n[WARNING] No text blocks found.")
            
    except Exception as e:
        print(f"\n[ERROR] An issue occurred during execution:")
        print(e)
        
    finally:
        try:
            print("Closing browser session...")
            driver.quit()
        except:
            pass

# --- Run Scraper ---
target_url = 'https://www.facebook.com/ikbmcrew/posts/pfbid02HezWcqFLWqFrqtLUdu4jRAptWnM1iXcNdNsPBMuGRYvHqmn9NYCFpX7MnPuVLeTTl'
scrape_with_selenium(target_url)

Preparing Edge Browser Settings...
Launching Edge Browser...
Navigating to https://www.facebook.com/ikbmcrew/posts/pfbid02HezWcqFLWqFrqtLUdu4jRAptWnM1iXcNdNsPBMuGRYvHqmn9NYCFpX7MnPuVLeTTl...

ACTION REQUIRED:
1. Look at the Edge browser.
2. If Facebook asks you to log in, take your time and do it.
3. Ensure the comments are fully visible on the screen.



Press ENTER in this text box right here when you are ready to scrape... 


Extracting elements...
Extracted 152 potential comment text blocks.

[SUCCESS] Saved data to facebook_comments2.csv


,id,message
0,1,今天到他陪我看Rice …
1,2,今天到他陪我看Rice …
2,3,"Dilaporkan Hilang Selepas Keluar Temu Duga, No..."
3,4,"Dilaporkan Hilang Selepas Keluar Temu Duga, No..."
4,5,Terbitan ikbm pada 28 Jun 2026: 07.24 pagi


Closing browser session...
